# Reproduce the archived coding audit

## tl;dr
The retained 4 × 5 coding table has Pearson discrepancy 57.2667 and Cramér’s V 0.39884. All 120 file-indexed rows remain semantically unresolved. General variable-status markers occur in 113 PDFs, but explicit reconfirmation markers occur in 111; these are whole-document text searches, not compliance findings.


## Context & Methods
This companion reruns all four permutation calculations (100,000 each) and both within-system bootstrap calculations (20,000 each), then checks the saved document-marker evidence and PDF identities. Run from the repository root or analysis directory after installing requirements.txt. The original archive is preserved at commit d345e7f391bef6f6c60c52d2a5907f0d166384ba.

### Key Assumptions
Coding labels are retained unchanged. Branch, page, and selection-rule provenance is unresolved. Exchangeability and the bootstrap sampling model are not established. Shared prompts alone do not prove dependence. No original respondent-level survey data are published. The PDF audit includes prompts and responses and does not verify speaker roles or executed checks.


In [1]:
from pathlib import Path
import contextlib, hashlib, io, json, subprocess, sys
root = Path.cwd()
if not (root / 'coding').is_dir():
    root = root.parent
assert (root / 'coding/final_outcome_coding_run_level.csv').is_file()
sys.path.insert(0, str(root / 'analysis'))
import reanalyze_outcomes as analysis


## Data
### 1. Validate the indexed coding table
The script checks 120 unique row IDs and paths, 30 records per system, valid categories, ranges, and unresolved semantic status. It does not validate the underlying coding against PDF passages.


In [2]:
rows = analysis.load_rows()
input_hash = hashlib.sha256(analysis.INPUT.read_bytes()).hexdigest()
assert all(row['semantic_verification_status'] == 'unresolved' for row in rows)
print({'rows': len(rows), 'source_unresolved': len(rows), 'input_sha256': input_hash})


{'rows': 120, 'source_unresolved': 120, 'input_sha256': '3cf4f67b7835a3286cc268606c2883132d231aab8b0cb1cc1cce17aee7dd5be9'}


## Results
### 2. Recompute the conditional analyses
The seed and resampling specifications are saved in reanalysis_results.json. This cell genuinely reruns the calculations and replaces their generated output file; it does not modify the coding inputs.


In [3]:
with contextlib.redirect_stdout(io.StringIO()):
    analysis.main()
results = json.loads(analysis.OUTPUT.read_text(encoding='utf-8'))
assert results['metadata']['input_csv_sha256'] == input_hash
for key, value in results.items():
    if isinstance(value, dict) and 'chi_square' in value:
        print(key, {'chi_square': value['chi_square'], 'V': value['cramers_v'], 'p_plus_one': value['permutation_p_plus_one'], 'exceedances': value['permutation_exceedances']})


primary_four_system_five_category {'chi_square': 57.26666666666667, 'V': 0.39884091323994597, 'p_plus_one': 9.99990000099999e-06, 'exceedances': 0}
sensitivity_three_standalone_five_category {'chi_square': 20.078947368421044, 'V': 0.33399058011018684, 'p_plus_one': 0.004949950500494995, 'exceedances': 494}
sensitivity_four_system_three_category {'chi_square': 37.0, 'V': 0.39264063297965823, 'p_plus_one': 1.999980000199998e-05, 'exceedances': 1}
sensitivity_three_standalone_three_category {'chi_square': 15.078947368421053, 'V': 0.2894338090558209, 'p_plus_one': 0.004099959000409996, 'exceedances': 409}


### 3. Check the document-marker evidence and file identities
The stored marker result records its extraction/cache route. This cell verifies every input PDF hash against that result. To rerun all text extraction directly from PDFs, execute `python analysis/audit_traceability.py` from the repository root; the next cell then reads the regenerated results. Prompt text can satisfy a marker; a label is not a validated parameter value.


In [4]:
audit = json.loads((root / 'analysis/traceability_audit_summary.json').read_text(encoding='utf-8'))
for relative, expected in audit['input_pdf_sha256'].items():
    assert hashlib.sha256((root / relative).read_bytes()).hexdigest() == expected, relative
assert len(audit['input_pdf_sha256']) == 120
print('PDF hashes checked:', len(audit['input_pdf_sha256']))
for key, value in audit['fields'].items():
    print(key, value['marker_present'], '/', value['denominator'])
assert audit['fields']['variable_status_label']['marker_present'] == 113
assert audit['fields']['explicit_reconfirm_variable_status']['marker_present'] == 111


PDF hashes checked: 120
readable_text 120 / 120
artifact_reference 120 / 120
explicit_result_log_label 119 / 120
all_four_state_dimensions 120 / 120
trigger_or_threshold 120 / 120
variable_status_label 113 / 120
explicit_reconfirm_variable_status 111 / 120
metadata_labels_present 0 / 120
independent_checker_phrase 0 / 120


## Takeaways
The arithmetic and file index are reproducible; source selection is not established. The two diagnostic discrepancies in docs/provenance_audit.md must not be extrapolated into an error rate or silently recoded. Original profiles contain dated cutoff violations and unresolved citation tokens (docs/input_artifact_caveats.md). See docs/replication_notes.md for figure generation, dependencies, and the distinction between current and legacy analyses.
